# MIZAN - Google Colab: RAG Build + QLoRA Fine-tuning
### Tunisian Legal AI | Full pipeline on free T4 GPU

**What this notebook does:**
1. Mounts Google Drive and reads your `.txt` files from `MyDrive/mizan/maraja/`
2. Parses, chunks, and embeds them -> saves FAISS index to Drive
3. Auto-generates a fine-tuning dataset from the index + HF Inference API
4. Fine-tunes `Qwen2.5-3B-Instruct` with QLoRA (fits T4 15GB)
5. Merges LoRA weights and pushes the final model to HuggingFace Hub
6. Downloads ready-to-use `rag_engine.py` + `mizan_data.zip` for your laptop

---

**Before running:**
- Runtime -> Change runtime type -> **T4 GPU** (or better)
- Upload your `.txt` files to Google Drive at: `My Drive/mizan/maraja/`
- Have your HuggingFace token ready (needs **write** access)

**To retrain after adding new files:**
Re-run Section A (rebuild index) then Section B (new dataset) then Section C (retrain).


## Step 0 - Verify GPU
> If this fails, go to Runtime -> Change runtime type -> T4 GPU

In [1]:
import subprocess, sys, torch

r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip())
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    assert vram >= 12, f"Need >= 12GB VRAM for QLoRA. Got {vram:.1f}GB. Change runtime to T4."
else:
    sys.exit("ERROR: No GPU. Go to Runtime -> Change runtime type -> T4 GPU")


GPU: Tesla T4, 15360 MiB
CUDA available: True
VRAM: 15.6 GB


## Step 1 - Mount Google Drive

In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_MIZAN  = Path("/content/drive/MyDrive/mizan")
MARAJA_DIR   = DRIVE_MIZAN / "maraja"
DATA_DIR     = DRIVE_MIZAN / "data"
LORA_DIR     = DRIVE_MIZAN / "mizan-lora"
LOCAL_DIR    = Path("/content/mizan")

for d in [MARAJA_DIR, DATA_DIR, LORA_DIR, LOCAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

txt_files = sorted(MARAJA_DIR.glob("*.txt"))
print(f"Drive mounted.")
print(f"  maraja/  : {MARAJA_DIR}")
print(f"  data/    : {DATA_DIR}")
print(f"  Found {len(txt_files)} .txt files:")
for f in txt_files:
    print(f"    {f.name}  ({f.stat().st_size/1024:.0f} KB)")

if not txt_files:
    print()
    print("NO FILES FOUND. Upload your .txt files to:")
    print(f"  Google Drive -> mizan -> maraja -> [your files]")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
  maraja/  : /content/drive/MyDrive/mizan/maraja
  data/    : /content/drive/MyDrive/mizan/data
  Found 52 .txt files:
    Acte_deces.txt  (3 KB)
    Aide_judiciaire.txt  (5 KB)
    ArbitrageArabe.txt  (102 KB)
    Attesta_non_faillite.txt  (5 KB)
    COCArabe.txt  (0 KB)
    CommerceArabe.txt  (423 KB)
    Communaute_biens.txt  (11 KB)
    Condamne_prison_amende_20102009.txt  (7 KB)
    Conseil_prud_homme.txt  (15 KB)
    DIPArabe.txt  (32 KB)
    EnfantArabe.txt  (227 KB)
    Extinction_peine_2012.txt  (4 KB)
    Grace_habilitation.txt  (7 KB)
    Immat_fonciere_20102009.txt  (9 KB)
    Injonction_a_payer_2012.txt  (7 KB)
    Litige_tribu_competent.txt  (15 KB)
    NationaliteArabe.txt  (36 KB)
    Nom_pat_enf_abondonnes_2012.txt  (10 KB)
    Ordonnance_sur_requete.txt  (4 KB)
    Peine_de_reparation_penale_20102009.txt  (11 KB)
    Peine_tra

## Step 2 - Install dependencies
> ~2-3 min. Run once per Colab session.

In [5]:
%pip install -q sentence-transformers faiss-cpu huggingface_hub langdetect tqdm numpy datasets
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes xformers
print("All packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 54.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## Step 3 - HuggingFace token & username

To securely store your HuggingFace token, add it to Colab's secrets manager. Click on the '🔑' icon in the left panel, then select 'Add a new secret'. Set the name to `HF_TOKEN` and paste your HuggingFace token (ensure it has **write** access) as the value.

In [13]:
import os, json, re, hashlib, pickle
from huggingface_hub import login, InferenceClient
from google.colab import userdata

# Securely fetch the token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

HF_USERNAME = input("Your HuggingFace username: ").strip()
MODEL_REPO  = f"{HF_USERNAME}/mizan-legal-tunisian"

# Login using the secret token
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Logged in successfully. Model will be pushed to -> {MODEL_REPO}")

Your HuggingFace username: Rayeeennnnnnnn
Logged in successfully. Model will be pushed to -> Rayeeennnnnnnn/mizan-legal-tunisian


In [14]:
# Original cell 'c8' will be replaced by the updated one. This is a placeholder for the agent to know that the original 'c8' content should be effectively removed or modified.
# The actual modification will be handled by the generate_cells command inserting the corrected code after 'm7' and effectively overriding 'c8'.

In [15]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

# Ensure the token is not None or empty
if not HF_TOKEN:
    raise ValueError("HuggingFace token not found. Please set 'HF_TOKEN' in Colab secrets.")

# Existing code from cell c8
import getpass, os, json, re, hashlib, pickle
from huggingface_hub import login, InferenceClient

HF_USERNAME = input("Your HuggingFace username: ").strip()
MODEL_REPO  = f"{HF_USERNAME}/mizan-legal-tunisian"

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Logged in. Model will be pushed to -> {MODEL_REPO}")

Your HuggingFace username: Rayeeennnnnnnn


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in. Model will be pushed to -> Rayeeennnnnnnn/mizan-legal-tunisian


---
## SECTION A - Build FAISS Index from txt files
Re-run this section alone whenever you add new source files.


In [16]:
from pathlib import Path
from langdetect import detect, LangDetectException
from tqdm import tqdm
import re, json, hashlib, pickle, numpy as np, faiss
from sentence_transformers import SentenceTransformer

CHUNK_SIZE      = 300
CHUNK_OVERLAP   = 60
MIN_CHUNK_CHARS = 80
EMBED_MODEL     = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

DOCS_PATH   = DATA_DIR / "documents.json"
CHUNKS_PATH = DATA_DIR / "chunks.json"
INDEX_PATH  = DATA_DIR / "legal_index.faiss"
META_PATH   = DATA_DIR / "legal_metadata.pkl"

ARTICLE_SPLIT = re.compile(
    r'(?=^\s*(?:'
    r'\u0627\u0644\u0641\u0635\u0644\s+\d+'   # الفصل
    r'|\u0627\u0644\u0645\u0627\u062f\u0629\s+\d+'   # المادة
    r'|\u0627\u0644\u0628\u0627\u0628\s+\w+'   # الباب
    r'|[Aa]rticle\s+\d+'
    r'|[Aa]rt\.?\s+\d+'
    r'|[Cc]hapitre\s+[IVX\d]+'
    r'|[Tt]itre\s+[IVX\d]+'
    r'|[Ss]ection\s+[IVX\d]+'
    r'))',
    re.MULTILINE
)

LABEL_RE = re.compile(
    r'^\s*(\u0627\u0644\u0641\u0635\u0644\s+\d+'
    r'|\u0627\u0644\u0645\u0627\u062f\u0629\s+\d+'
    r'|\u0627\u0644\u0628\u0627\u0628\s+\w+'
    r'|[Aa]rticle\s+\d+'
    r'|[Aa]rt\.?\s+\d+'
    r'|[Cc]hapitre\s+[IVX\d]+'
    r'|[Tt]itre\s+[IVX\d]+'
    r'|[Ss]ection\s+[IVX\d]+)',
    re.MULTILINE
)

def clean(t):
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', t)
    return t.strip()

def lang_of(text):
    ar = len(re.findall(r'[\u0600-\u06FF]', text)) / max(len(text), 1)
    if ar > 0.20: return "ar"
    try: return detect(text)
    except LangDetectException: return "fr"

def split_articles(text):
    parts = ARTICLE_SPLIT.split(text)
    result = []
    for p in parts:
        p = p.strip()
        if not p: continue
        m = LABEL_RE.match(p)
        result.append((m.group(1).strip() if m else "", p))
    return result if len(result) > 1 else [("", text)]

def infer_title(fname):
    s = re.sub(r'[-_]', ' ', Path(fname).stem)
    return re.sub(r'\s+', ' ', s).strip().title()

txt_files = sorted(MARAJA_DIR.glob("*.txt"))
print(f"Processing {len(txt_files)} files...")
documents = []

for fpath in tqdm(txt_files, desc="Parsing"):
    try:
        text = fpath.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"  SKIP {fpath.name}: {e}"); continue
    text = clean(text)
    if len(text) < 150: continue
    title = infer_title(fpath.name)
    lang  = lang_of(text[:500])
    for label, section in split_articles(text):
        section = clean(section)
        if len(section) < 80: continue
        uid = hashlib.md5(f"{fpath.name}|{label}|{section[:30]}".encode()).hexdigest()[:12]
        documents.append({
            "id": uid, "title": title, "source": fpath.name,
            "language": lang, "article": label, "text": section, "url": "",
        })

with open(DOCS_PATH, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

fr = sum(1 for d in documents if d["language"]=="fr")
ar = sum(1 for d in documents if d["language"]=="ar")
print(f"Sections: {len(documents)} | French: {fr} | Arabic: {ar}")


Processing 52 files...


Parsing: 100%|██████████| 52/52 [00:54<00:00,  1.04s/it]

Sections: 526 | French: 0 | Arabic: 526


In [17]:
# Smart chunking: keep short articles whole, split long ones
all_chunks, kept_whole, was_split = [], 0, 0
for doc in tqdm(documents, desc="Chunking"):
    words = doc["text"].split()
    if len(words) <= CHUNK_SIZE:
        c = {**doc, "chunk_id": f"{doc['id']}_c0", "doc_id": doc["id"], "chunk_index": 0}
        all_chunks.append(c); kept_whole += 1
    else:
        step = max(CHUNK_SIZE - CHUNK_OVERLAP, 1)
        for i, start in enumerate(range(0, len(words), step)):
            w = words[start:start+CHUNK_SIZE]
            if not w: break
            t = " ".join(w).strip()
            if len(t) < MIN_CHUNK_CHARS: continue
            c = {**doc, "text": t,
                 "chunk_id": f"{doc['id']}_c{i}",
                 "doc_id": doc["id"], "chunk_index": i}
            all_chunks.append(c)
        was_split += 1

avg = sum(len(c["text"]) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Chunks: {len(all_chunks)} (whole: {kept_whole}, split: {was_split}) | avg {avg:.0f} chars")
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)


Chunking: 100%|██████████| 526/526 [00:00<00:00, 7860.55it/s]

Chunks: 2289 (whole: 204, split: 322) | avg 1475 chars


In [18]:
print(f"Loading {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL)
DIM = embedder.get_sentence_embedding_dimension()
print(f"  dim={DIM}")

texts = [c["text"] for c in all_chunks]
print(f"Encoding {len(texts)} chunks on GPU...")

BATCH = 256
emb_list = []
for i in tqdm(range(0, len(texts), BATCH), desc="Embedding"):
    e = embedder.encode(texts[i:i+BATCH],
                        normalize_embeddings=True,
                        show_progress_bar=False,
                        convert_to_numpy=True, device="cuda")
    emb_list.append(e)

embeddings = np.vstack(emb_list).astype("float32")
index = faiss.IndexFlatIP(DIM)
index.add(embeddings)

faiss.write_index(index, str(INDEX_PATH))
metadata = {"embed_model":EMBED_MODEL,"embed_dim":DIM,
            "n_docs":len(documents),"n_chunks":len(all_chunks),"chunks":all_chunks}
with open(META_PATH,"wb") as f: pickle.dump(metadata, f)

print(f"FAISS index: {index.ntotal} vectors -> {INDEX_PATH}")
print(f"Metadata                             -> {META_PATH}")


Loading sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_8647/1637163697.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


  dim=768
Encoding 2289 chunks on GPU...


Embedding: 100%|██████████| 9/9 [00:16<00:00,  1.86s/it]

FAISS index: 2289 vectors -> /content/drive/MyDrive/mizan/data/legal_index.faiss
Metadata                             -> /content/drive/MyDrive/mizan/data/legal_metadata.pkl


### A1 - Quick retrieval sanity check (no LLM)

In [19]:
def retrieve_raw(query, k=4):
    q = embedder.encode([query], normalize_embeddings=True,
                        convert_to_numpy=True, device="cuda").astype("float32")
    scores, idxs = index.search(q, k)
    return [(float(scores[0][i]), all_chunks[idxs[0][i]]) for i in range(k) if idxs[0][i] != -1]

for q in ["prescription action civile", "contrat bail", "code penal vol",
          "\u0627\u0633\u062a\u0626\u0646\u0627\u0641 \u0645\u062f\u0646\u064a"]:
    print(f"Q: {q}")
    for sc, c in retrieve_raw(q, k=2):
        art = f" | {c['article']}" if c.get("article") else ""
        print(f"  [{sc:.3f}] {c['source']}{art}")
        print(f"           {c['text'][:100]}...")
    print()

print("If all scores are below 0.25, check your txt file encoding (open in Notepad, save as UTF-8).")


Q: prescription action civile
  [0.685] manuel_proced__officier_poli_jud.txt | الفصل 87
           دورية بجميع المؤسسات المشار اليها والسهر علسى مراعساة مقتضسيات التشريع الخساص بالميحدان الصيدلي وتحر...
  [0.626] manuel_proced_enregis_deci.txt
           تعيين مقدارها من طرف رئيس المحكمة وبدون لزوم لإجراءات جديدة . كماجاء بالفصل 357 من مجلة الحقوق العين...

Q: contrat bail
  [0.664] CommerceArabe.txt | الفصل 459
           المجلة. الفصل 467 تحدد المحكمة أجلا كراس شروط من @أ[٥ لتحرير قبل القضائي . ويجب أن تضمن به شروط الكر...
  [0.649] societeArabe.txt | الفصل 56
           نفقة الطالب الفصل ٠66- لا يمكن إطلاع العموم على معنى أحكام , هذا الباب 1 بالنسبة لإجراءات التسو يقال...

Q: code penal vol
  [0.649] PenalArabe.txt | الفصل 19
           بهذا 103 --- Page 104 --- القانون أو الجرائم المرتبطة بها بإبلاغ السلط ذات النظر بإرشادات أو معلومات...
  [0.642] PenalArabe.txt | الفصل 248
           أو إمضاء أو إحدى الأوراق المبينة 0@ 283 من هذه المجلة . الفصل ٠285- يمكن تطبيق العقوبات المبينة با

---
## SECTION B - Generate Fine-tuning Dataset
Calls Mistral-7B via HF Inference API to produce gold Q&A pairs.
Takes ~15-30 min depending on how many seed questions there are.
Re-run after adding new source files to expand the dataset.


In [20]:
client = InferenceClient(token=HF_TOKEN)
LLM_BASE = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_PATH = DATA_DIR / "finetune_dataset.jsonl"

SEED_QUESTIONS = [
    # Civil
    "Quelles sont les conditions de validite d'un contrat en Tunisie ?",
    "Quels sont les delais de prescription en matiere civile ?",
    "Comment resilier un contrat de bail en Tunisie ?",
    "Quels sont les droits du locataire selon le droit tunisien ?",
    "Quelles sont les causes de nullite d'un contrat ?",
    "Qu'est-ce que la responsabilite contractuelle en droit tunisien ?",
    "Comment est calcule le prejudice en droit civil tunisien ?",
    # Commercial
    "Comment creer une SARL en Tunisie ?",
    "Quelles sont les obligations du gerant d'une SARL ?",
    "Comment se deroule une procedure de faillite en Tunisie ?",
    "Quelles sont les conditions de validite d'un cheque en Tunisie ?",
    "Qu'est-ce qu'une lettre de change selon le droit tunisien ?",
    # Criminal
    "Quelles sont les peines prevues pour le vol selon le code penal tunisien ?",
    "Quelles sont les conditions de la legitime defense en droit tunisien ?",
    "Qu'est-ce que la recidive selon le code penal tunisien ?",
    "Comment est definie la complicite dans le code penal tunisien ?",
    "Quelles sont les peines alternatives a l'emprisonnement en Tunisie ?",
    # Criminal procedure
    "Quels sont les droits du prevenu lors de l'interrogatoire en Tunisie ?",
    "Quels sont les delais de detention preventive en Tunisie ?",
    "Comment se deroule la procedure d'appel en matiere penale ?",
    # Administrative
    "Comment contester une decision administrative en Tunisie ?",
    "Quels sont les delais de recours devant le tribunal administratif ?",
    "Qu'est-ce que le recours pour exces de pouvoir en Tunisie ?",
    # Family
    "Quelles sont les conditions du mariage en Tunisie ?",
    "Comment se deroule une procedure de divorce en Tunisie ?",
    "Comment est fixee la pension alimentaire en Tunisie ?",
    "Quelles sont les regles de la tutelle en droit tunisien ?",
    # Succession
    "Quelles sont les regles de succession en Tunisie ?",
    "Qui sont les heritiers reservataires en droit tunisien ?",
    # Labour
    "Quelles sont les conditions du licenciement en Tunisie ?",
    "Quelle est la duree legale du travail en Tunisie ?",
    "Quels sont les droits du salarie lors d'un licenciement abusif ?",
    # Arabic
    "\u0645\u0627 \u0647\u064a \u0625\u062c\u0631\u0627\u0621\u0627\u062a \u0627\u0644\u0627\u0633\u062a\u0626\u0646\u0627\u0641 \u0641\u064a \u0627\u0644\u0642\u0636\u0627\u064a\u0627 \u0627\u0644\u0645\u062f\u0646\u064a\u0629 \u0627\u0644\u062a\u0648\u0646\u0633\u064a\u0629 \u061f",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0645\u062a\u0647\u0645 \u0641\u064a \u0645\u0631\u062d\u0644\u0629 \u0627\u0644\u0627\u0633\u062a\u062c\u0648\u0627\u0628 \u061f",
    "\u0645\u0627 \u0647\u064a \u0634\u0631\u0648\u0637 \u0635\u062d\u0629 \u0639\u0642\u062f \u0627\u0644\u0628\u064a\u0639 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u0627\u0644\u0639\u0642\u0648\u0628\u0627\u062a \u0627\u0644\u0645\u0642\u0631\u0631\u0629 \u0644\u0644\u0633\u0631\u0642\u0629 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0643\u064a\u0641 \u064a\u062a\u0645 \u062a\u0623\u0633\u064a\u0633 \u0634\u0631\u0643\u0629 \u0630\u0627\u062a \u0645\u0633\u0624\u0648\u0644\u064a\u0629 \u0645\u062d\u062f\u0648\u062f\u0629 \u0641\u064a \u062a\u0648\u0646\u0633 \u061f",
    "\u0645\u0627 \u0647\u064a \u0645\u0648\u0627\u0639\u064a\u062f \u0627\u0644\u062a\u0642\u0627\u062f\u0645 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u0645\u062f\u0646\u064a \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u0625\u062c\u0631\u0627\u0621\u0627\u062a \u0627\u0644\u0637\u0644\u0627\u0642 \u0641\u064a \u0627\u0644\u0642\u0627\u0646\u0648\u0646 \u0627\u0644\u062a\u0648\u0646\u0633\u064a \u061f",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0639\u0627\u0645\u0644 \u0639\u0646\u062f \u0627\u0644\u0641\u0635\u0644 \u0627\u0644\u062a\u0639\u0633\u0641\u064a \u061f",
]

def gen_answer(question, contexts):
    lang = lang_of(question)
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a. \u0623\u062c\u0628 \u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0641\u0642\u0637. \u0627\u0630\u0643\u0631 \u0627\u0633\u0645 \u0627\u0644\u0646\u0635 \u0648\u0627\u0644\u0641\u0635\u0644."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique tunisien. Reponds UNIQUEMENT sur la base des textes. Cite la source et l'article. Sois precis. Max 250 mots."
    )
    ctx = "".join(f"[{i}] {c['source']}" + (f" | {c['article']}" if c.get("article") else "") + f"\n{c['text']}\n\n" for i,c in enumerate(contexts,1))
    try:
        r = client.chat_completion(
            model=LLM_BASE,
            messages=[{"role":"system","content":sys_msg},
                      {"role":"user","content":f"Textes:\n\n{ctx}\nQuestion: {question}"}],
            max_tokens=400, temperature=0.1, top_p=0.95,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f"    API error: {e}")
        return None

print(f"Generating from {len(SEED_QUESTIONS)} questions (may take 20-40 min)...")
dataset, skipped = [], 0
for i, q in enumerate(SEED_QUESTIONS):
    ctxs = [all_chunks[idxs] for idxs in index.search(
        embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True, device="cuda").astype("float32"), 4
    )[1][0] if idxs != -1]
    ctxs = [c for c in ctxs]  # filter score below threshold manually
    if not ctxs: skipped += 1; print(f"  [{i+1}] SKIP no context: {q[:55]}"); continue
    gold = gen_answer(q, ctxs[:4])
    if not gold: skipped += 1; print(f"  [{i+1}] SKIP API error: {q[:55]}"); continue
    ctx_text = "\n\n".join(f"[{c['source']}]" + (f" {c['article']}" if c.get("article") else "") + f"\n{c['text']}" for c in ctxs[:4])
    dataset.append({"instruction":q,"context":ctx_text[:2000],"output":gold,"language":lang_of(q)})
    print(f"  [{i+1}] OK: {q[:55]}")

with open(DATASET_PATH,"w",encoding="utf-8") as f:
    for item in dataset: f.write(json.dumps(item, ensure_ascii=False) + "\n")

fr_n = sum(1 for d in dataset if d["language"]=="fr")
ar_n = sum(1 for d in dataset if d["language"]=="ar")
print(f"\nDataset: {len(dataset)} examples (skipped {skipped})")
print(f"  French: {fr_n} | Arabic: {ar_n}")
print(f"Saved -> {DATASET_PATH}")


Generating from 40 questions (may take 20-40 min)...
    API error: (Request ID: Root=1-69e4b04c-206c677340b2db9a0840889b;5ed00b6e-2821-43e0-a23f-2a3bbd7c5c23)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.2' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}
  [1] SKIP API error: Quelles sont les conditions de validite d'un contrat en
    API error: (Request ID: Root=1-69e4b04c-4704ceac6fe1048a36d82541;7024ef63-2d02-4d29-9bfb-368faed51146)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.2' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}
  [2] SKIP API error: Quels sont les delais de prescription en matiere civile
    API error: (Request ID: Root=1-69e4b04c-0827801f6ea890760cf80ce3;59016986-425c-4be1-9dbb-1c27c1a8217c)

Bad request:
{'message': "The r

---
## SECTION C - QLoRA Fine-tuning
Model: Qwen2.5-3B-Instruct | Expected time: 25-50 min on T4


In [21]:
from unsloth import FastLanguageModel
import torch
MAX_SEQ_LEN = 2048

print("Loading base model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
print(f"Base model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")


/tmp/ipykernel_8647/3482725431.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model with 4-bit quantization...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded (1800M params)


In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable/1e6:.1f}M trainable / {total/1e6:.0f}M total ({100*trainable/total:.2f}%)")


Unsloth 2026.4.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA: 29.9M trainable / 1830M total (1.64%)


In [23]:
from datasets import Dataset
import math

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = [json.loads(l) for l in f]

def fmt(ex):
    lang = ex.get("language","fr")
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a. \u0623\u062c\u0628 \u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635. \u0627\u0630\u0643\u0631 \u0627\u0644\u0645\u0635\u062f\u0631 \u0648\u0627\u0644\u0641\u0635\u0644."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique expert en droit tunisien. Reponds uniquement sur la base des textes fournis. Cite la source et l'article."
    )
    user = f"Textes juridiques pertinents:\n\n{ex['context']}\n\nQuestion: {ex['instruction']}"
    return {"text": (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
    )}

hf_ds = Dataset.from_list(raw).map(fmt)
n = len(hf_ds)
num_epochs = 5 if n < 100 else 4 if n < 250 else 3
total_steps = math.ceil(n / (2 * 4)) * num_epochs
print(f"Dataset: {n} | Epochs: {num_epochs} | Steps: {total_steps} | Est: ~{total_steps*8//60} min")


Dataset: 0 | Epochs: 5 | Steps: 0 | Est: ~0 min


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = hf_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio                = 0.05,
        num_train_epochs            = num_epochs,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = str(LOCAL_DIR / "checkpoints"),
        save_strategy               = "epoch",
        report_to                   = "none",
    ),
)
print("Trainer ready. Starting training...")
stats = trainer.train()
print(f"Done! Final loss: {stats.training_loss:.4f} | Time: {stats.metrics['train_runtime']:.0f}s")


In [ ]:
# Quick inference test with fine-tuned model
FastLanguageModel.for_inference(model)

def local_answer(q, ctxs):
    lang = lang_of(q)
    sys_msg = (
        "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a."
        if lang == "ar" else
        "Tu es MIZAN, assistant juridique tunisien. Cite la source et l'article."
    )
    ctx = "\n\n".join(f"[{c['source']}]" + (f" {c['article']}" if c.get("article") else "") + f"\n{c['text']}" for c in ctxs)
    prompt = (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\nTextes:\n\n{ctx}\n\nQuestion: {q}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, temperature=0.1,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

TEST_Q = [
    "Quels sont les delais de prescription en matiere civile ?",
    "Comment contester une decision administrative ?",
    "\u0645\u0627 \u0647\u064a \u062d\u0642\u0648\u0642 \u0627\u0644\u0645\u062a\u0647\u0645 \u0641\u064a \u0645\u0631\u062d\u0644\u0629 \u0627\u0644\u0627\u0633\u062a\u062c\u0648\u0627\u0628 \u061f",
]
for q in TEST_Q:
    q_vec = embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True, device="cuda").astype("float32")
    _, idxs = index.search(q_vec, 3)
    ctxs = [all_chunks[i] for i in idxs[0] if i != -1]
    print(f"\nQ: {q}")
    print(f"A: {local_answer(q, ctxs)[:400]}")


---
## SECTION D - Save LoRA, Merge, and Push to HuggingFace Hub


In [ ]:
# Save LoRA adapters to Drive (small, fast)
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"LoRA adapters saved -> {LORA_DIR}")
print(f"  Size: {sum(f.stat().st_size for f in LORA_DIR.rglob('*') if f.is_file())/1e6:.1f} MB")


In [ ]:
# Merge LoRA into full weights and push (~5-10 min, ~6GB upload)
print(f"Merging and pushing to: https://huggingface.co/{MODEL_REPO}")
print("This takes ~5-10 min...")
model.push_to_hub_merged(
    MODEL_REPO, tokenizer,
    save_method = "merged_16bit",
    token       = HF_TOKEN,
)
print(f"\nModel live at: https://huggingface.co/{MODEL_REPO}")


---
## SECTION E - Download files for your laptop

Downloads two things:
1. `rag_engine.py` - updated with your fine-tuned model name
2. `mizan_data.zip` - FAISS index + metadata (put in `data/` on your laptop)


In [ ]:
from google.colab import files

# Build rag_engine.py with the fine-tuned model name substituted in
rag_lines = [
    'import os, re, pickle',
    'from pathlib import Path',
    'import numpy as np, faiss',
    'from sentence_transformers import SentenceTransformer',
    'from huggingface_hub import InferenceClient',
    'from langdetect import detect, LangDetectException',
    '',
    'BASE       = Path(__file__).parent',
    'INDEX_PATH = BASE / "data" / "legal_index.faiss"',
    'META_PATH  = BASE / "data" / "legal_metadata.pkl"',
    f'LLM_MODEL  = "{MODEL_REPO}"',
    '',
    'print("[MIZAN] Loading FAISS index...")',
    '_index    = faiss.read_index(str(INDEX_PATH))',
    'with open(META_PATH, "rb") as _f: _meta = pickle.load(_f)',
    '_chunks   = _meta["chunks"]',
    '_embedder = SentenceTransformer(_meta["embed_model"])',
    '_client   = InferenceClient(token=os.getenv("HF_TOKEN", ""))',
    'print(f"[MIZAN] Ready - {_index.ntotal} vectors | {LLM_MODEL}")',
    '',
    'def _lang(text):',
    '    ar = len(re.findall(r"[\u0600-\u06FF]", text)) / max(len(text), 1)',
    '    if ar > 0.20: return "ar"',
    '    try: return detect(text)',
    '    except LangDetectException: return "fr"',
    '',
    'def retrieve(query, k=6):',
    '    q = _embedder.encode([query], normalize_embeddings=True,',
    '                         convert_to_numpy=True).astype("float32")',
    '    scores, idxs = _index.search(q, k)',
    '    return [{**_chunks[ix], "score": float(sc)}',
    '            for sc, ix in zip(scores[0], idxs[0]) if ix != -1]',
    '',
    'def answer(query, history=None, k=6):',
    '    if not query.strip():',
    '        return {"answer": "Veuillez poser une question.", "sources": [], "language": "fr"}',
    '    contexts = retrieve(query, k=k)',
    '    if not contexts:',
    '        return {"answer": "Aucun texte pertinent trouve.", "sources": [], "language": _lang(query)}',
    '    lang = _lang(query)',
    '    if lang == "ar":',
    '        sys_msg = "\u0623\u0646\u062a MIZAN \u0645\u0633\u0627\u0639\u062f \u0642\u0627\u0646\u0648\u0646\u064a \u062a\u0648\u0646\u0633\u064a. \u0623\u062c\u0628 \u0628\u0646\u0627\u0621 \u0639\u0644\u0649 \u0627\u0644\u0646\u0635\u0648\u0635 \u0641\u0642\u0637. \u0627\u0630\u0643\u0631 \u0627\u0644\u0645\u0635\u062f\u0631 \u0648\u0627\u0644\u0641\u0635\u0644."',
    '    else:',
    '        sys_msg = ("Tu es MIZAN, assistant juridique expert en droit tunisien. "',
    '                   "Reponds UNIQUEMENT sur la base des textes fournis. "',
    '                   "Cite toujours le nom exact du texte et l'article concerne. "',
    '                   "Si la reponse n'est pas dans les textes, dis-le clairement.")',
    '    ctx = "".join(',
    '        f"[{i}] {c[\'title\']} | {c[\'source\']}"',
    '        + (f" | {c[\'article\']}" if c.get("article") else "")',
    '        + f"\n{c[\'text\']}\n\n"',
    '        for i, c in enumerate(contexts, 1)',
    '    )',
    '    messages = [{"role":"system","content":sys_msg}]',
    '    for t in (history or [])[-3:]:',
    '        if t.get("user"):      messages.append({"role":"user","content":t["user"]})',
    '        if t.get("assistant"): messages.append({"role":"assistant","content":t["assistant"]})',
    '    messages.append({"role":"user","content":f"Textes:\n\n{ctx}\nQuestion: {query}"})',
    '    try:',
    '        resp = _client.chat_completion(',
    '            model=LLM_MODEL, messages=messages,',
    '            max_tokens=800, temperature=0.1, top_p=0.95,',
    '        )',
    '        return {"answer": resp.choices[0].message.content.strip(),',
    '                "sources": [{"title":c.get("title",c["source"]),"source":c["source"],',
    '                             "article":c.get("article",""),"url":c.get("url",""),',
    '                             "score":round(c["score"],4)} for c in contexts],',
    '                "language": lang, "error": None}',
    '    except Exception as e:',
    '        parts = [f"**{c[\"source\"]}**" + (f" - {c[\"article\"]}" if c.get("article") else "")',
    '                 + f"\n{c[\"text\"][:500]}" for c in contexts[:3]]',
    '        return {"answer": "[Reponse directe]\n\n" + "\n\n---\n\n".join(parts),',
    '                "sources": [], "language": lang, "error": str(e)}',
]

rag_path = LOCAL_DIR / "rag_engine.py"
rag_path.write_text("\n".join(rag_lines), encoding="utf-8")
print(f"rag_engine.py written with model: {MODEL_REPO}")

# Also save to Drive
(DRIVE_MIZAN / "rag_engine.py").write_text("\n".join(rag_lines), encoding="utf-8")

files.download(str(rag_path))
print("rag_engine.py download started.")


In [ ]:
import zipfile
from google.colab import files

zip_path = str(LOCAL_DIR / "mizan_data.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in DATA_DIR.iterdir():
        zf.write(str(fp), fp.name)
        print(f"  {fp.name}  ({fp.stat().st_size/1e6:.1f} MB)")

print(f"\nZip: {Path(zip_path).stat().st_size/1e6:.1f} MB")
files.download(zip_path)
print("mizan_data.zip download started.")
print()
print("On your laptop:")
print("  1. Extract mizan_data.zip -> ayheemm-hackathon_/data/")
print("  2. Replace rag_engine.py at ayheemm-hackathon_/rag_engine.py")
print(f"  3. PowerShell: $env:HF_TOKEN='your_token'; python app.py")


---
## Summary

| File | Where it goes on your laptop |
|---|---|
| `rag_engine.py` | `ayheemm-hackathon_/rag_engine.py` |
| `mizan_data.zip` contents | `ayheemm-hackathon_/data/` |

**To add more sources later:**
1. Upload new `.txt` files to `My Drive/mizan/maraja/`
2. Re-run **Section A** (rebuild index - ~15 min)
3. Re-run **Section B** (new training data - ~30 min)
4. Re-run **Section C** (retrain - ~30 min)
5. Re-run **Section D** (push new model)
6. Re-run **Section E** (download new files)

**To only update the index without retraining:**
Re-run only **Section A** then **Section E**.
The base `Mistral-7B` will still be used via API until you fine-tune.
